# Chapter 5.6: Knowledge Graph Enhanced Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Construct a **knowledge graph (KG)** with entities, relations, and their embeddings
2. Implement KG embedding methods: **TransE**, **TransR**, and **RotatE** for recommendation
3. Implement **RippleNet**: preference propagation on knowledge graphs
4. Understand **KGAT** (Knowledge Graph Attention Network) and attentive KG propagation
5. Design **KGIN** with disentangled user intent modeling
6. Integrate side information from KGs to improve cold-start recommendation
7. Evaluate KG-enhanced recommenders and analyze the effect of KG connectivity

## Prerequisites

- Chapter 5.5 (GNN basics for recommendation)
- Basic knowledge graph concepts (triples, entities, relations)
- PyTorch embedding layers and loss functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part5/chapter_5.6_kg_rec.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part5/chapter_5.6_kg_rec.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random
from collections import defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Knowledge Graphs for Recommendation

A **knowledge graph (KG)** enriches the recommendation system with structured side information:

- **Entities**: items, categories, brands, actors, directors, etc.
- **Relations**: "belongs_to", "directed_by", "similar_to", etc.
- **Triples**: (head_entity, relation, tail_entity), e.g., ("Inception", "directed_by", "Nolan")

Benefits for recommendation:
1. **Alleviates data sparsity**: KG provides additional connections
2. **Cold-start**: New items can be linked via KG relations
3. **Explainability**: Recommendation paths through the KG

> **💡 Concept:** In KG-enhanced rec, items in the recommendation system are aligned with entities in the KG. The KG provides a "semantic bridge" between items that may not have direct co-interaction patterns.

## 2. Synthetic Knowledge Graph Data

In [ ]:
def generate_kg_rec_data(n_users=400, n_items=200, n_entities=500, n_relations=10,
                         n_interactions=5000, n_triples=3000, seed=42):
    """Generate synthetic user-item interactions and knowledge graph."""
    rng = np.random.RandomState(seed)
    
    # Items are a subset of entities (entities 0..n_items-1)
    # Additional entities: categories, brands, attributes
    
    # Assign items to categories (entities n_items..n_items+20)
    n_categories = 20
    item_category = rng.randint(0, n_categories, size=n_items)
    
    # Generate user-item interactions
    user_items = defaultdict(set)
    interactions = []
    user_prefs = rng.randint(0, n_categories, size=(n_users, 3))  # Each user likes 3 cats
    
    while len(interactions) < n_interactions:
        u = rng.randint(0, n_users)
        if rng.random() < 0.7:
            cat = rng.choice(user_prefs[u])
            cat_items = np.where(item_category == cat)[0]
            if len(cat_items) > 0:
                item = rng.choice(cat_items)
            else:
                item = rng.randint(0, n_items)
        else:
            item = rng.randint(0, n_items)
        
        if item not in user_items[u]:
            user_items[u].add(int(item))
            interactions.append((u, int(item)))
    
    # Generate KG triples
    relation_names = [
        "belongs_to_category", "has_brand", "has_attribute", "similar_to",
        "also_bought", "also_viewed", "produced_by", "has_genre",
        "has_color", "has_material"
    ]
    
    triples = []
    # Item -> Category triples
    for item in range(n_items):
        cat_entity = n_items + item_category[item]
        triples.append((item, 0, cat_entity))  # belongs_to_category
    
    # Random KG triples
    while len(triples) < n_triples:
        head = rng.randint(0, n_entities)
        rel = rng.randint(0, n_relations)
        tail = rng.randint(0, n_entities)
        if head != tail:
            triples.append((int(head), int(rel), int(tail)))
    
    return {
        "interactions": interactions,
        "triples": triples,
        "n_users": n_users,
        "n_items": n_items,
        "n_entities": n_entities,
        "n_relations": n_relations,
        "user_items": user_items,
        "item_category": item_category,
        "relation_names": relation_names,
    }

data = generate_kg_rec_data()
print(f"Users: {data['n_users']}, Items: {data['n_items']}, Entities: {data['n_entities']}")
print(f"Interactions: {len(data['interactions'])}, KG Triples: {len(data['triples'])}")
print(f"Relations: {data['relation_names']}")
print(f"Sample triples: {data['triples'][:5]}")

## 3. KG Embedding Methods: TransE, TransR, RotatE

### TransE (Bordes et al., 2013)
Models relations as translations: $\mathbf{h} + \mathbf{r} \approx \mathbf{t}$

$$\mathcal{L} = \sum_{(h,r,t)} \left[ \|\mathbf{h} + \mathbf{r} - \mathbf{t}\| - \|\mathbf{h}' + \mathbf{r} - \mathbf{t}'\| + \gamma \right]_+$$

### TransR (Lin et al., 2015)
Projects entities to relation-specific spaces: $\mathbf{M}_r \mathbf{h} + \mathbf{r} \approx \mathbf{M}_r \mathbf{t}$

### RotatE (Sun et al., 2019)
Models relations as rotations in complex space: $\mathbf{h} \circ \mathbf{r} \approx \mathbf{t}$

where $\circ$ is element-wise complex multiplication and $|r_i| = 1$.

In [ ]:
class TransE(nn.Module):
    """TransE: Translating Embeddings (Bordes et al., 2013)."""
    
    def __init__(self, n_entities, n_relations, embed_dim=64, margin=1.0):
        super().__init__()
        self.entity_emb = nn.Embedding(n_entities, embed_dim)
        self.relation_emb = nn.Embedding(n_relations, embed_dim)
        self.margin = margin
        
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        # Normalize relation embeddings
        with torch.no_grad():
            self.relation_emb.weight.data = F.normalize(self.relation_emb.weight.data, dim=-1)
    
    def score(self, head, relation, tail):
        """Compute distance score: ||h + r - t||."""
        h = self.entity_emb(head)
        r = self.relation_emb(relation)
        t = self.entity_emb(tail)
        return torch.norm(h + r - t, p=2, dim=-1)
    
    def loss(self, head, relation, tail, neg_tail):
        """Margin-based ranking loss."""
        pos_score = self.score(head, relation, tail)
        neg_score = self.score(head, relation, neg_tail)
        return F.relu(pos_score - neg_score + self.margin).mean()

class RotatE(nn.Module):
    """RotatE: Knowledge Graph Embedding by Relational Rotation (Sun et al., 2019)."""
    
    def __init__(self, n_entities, n_relations, embed_dim=64, margin=6.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.margin = margin
        
        # Complex embeddings: real and imaginary parts
        self.entity_re = nn.Embedding(n_entities, embed_dim)
        self.entity_im = nn.Embedding(n_entities, embed_dim)
        # Relation: phase angles (unit complex numbers)
        self.relation_phase = nn.Embedding(n_relations, embed_dim)
        
        nn.init.xavier_uniform_(self.entity_re.weight)
        nn.init.xavier_uniform_(self.entity_im.weight)
        nn.init.uniform_(self.relation_phase.weight, -np.pi, np.pi)
    
    def score(self, head, relation, tail):
        """Compute rotation distance: ||h * r - t|| in complex space."""
        h_re = self.entity_re(head)
        h_im = self.entity_im(head)
        t_re = self.entity_re(tail)
        t_im = self.entity_im(tail)
        
        phase = self.relation_phase(relation)
        r_re = torch.cos(phase)
        r_im = torch.sin(phase)
        
        # Complex multiplication: (h_re + i*h_im) * (r_re + i*r_im)
        rot_re = h_re * r_re - h_im * r_im
        rot_im = h_re * r_im + h_im * r_re
        
        # Distance to tail
        diff_re = rot_re - t_re
        diff_im = rot_im - t_im
        
        return torch.sqrt(diff_re ** 2 + diff_im ** 2 + 1e-8).sum(dim=-1)
    
    def loss(self, head, relation, tail, neg_tail):
        pos_score = self.score(head, relation, tail)
        neg_score = self.score(head, relation, neg_tail)
        return F.relu(pos_score - neg_score + self.margin).mean()

# Test
transe = TransE(data["n_entities"], data["n_relations"], embed_dim=64)
rotate = RotatE(data["n_entities"], data["n_relations"], embed_dim=64)

h = torch.tensor([0, 1, 2])
r = torch.tensor([0, 1, 2])
t = torch.tensor([10, 20, 30])
print(f"TransE scores: {transe.score(h, r, t).detach().numpy()}")
print(f"RotatE scores: {rotate.score(h, r, t).detach().numpy()}")

In [ ]:
# Train TransE on synthetic KG
class KGDataset(Dataset):
    def __init__(self, triples, n_entities):
        self.triples = triples
        self.n_entities = n_entities
    
    def __len__(self):
        return len(self.triples)
    
    def __getitem__(self, idx):
        h, r, t = self.triples[idx]
        neg_t = random.randint(0, self.n_entities - 1)
        return (
            torch.tensor(h, dtype=torch.long),
            torch.tensor(r, dtype=torch.long),
            torch.tensor(t, dtype=torch.long),
            torch.tensor(neg_t, dtype=torch.long),
        )

kg_dataset = KGDataset(data["triples"], data["n_entities"])
kg_loader = DataLoader(kg_dataset, batch_size=256, shuffle=True)

# Train TransE
torch.manual_seed(SEED)
transe = TransE(data["n_entities"], data["n_relations"], embed_dim=64)
optimizer = torch.optim.Adam(transe.parameters(), lr=0.001)

kg_losses = []
for epoch in range(20):
    total_loss = 0
    n_b = 0
    for h, r, t, neg_t in kg_loader:
        loss = transe.loss(h, r, t, neg_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Re-normalize entity embeddings
        with torch.no_grad():
            transe.entity_emb.weight.data = F.normalize(transe.entity_emb.weight.data, dim=-1)
        total_loss += loss.item()
        n_b += 1
    kg_losses.append(total_loss / n_b)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/20 — KG Loss: {kg_losses[-1]:.4f}")

## 4. RippleNet: Preference Propagation on KG

**RippleNet** (Wang et al., CIKM 2018) propagates user preferences on the KG like ripples on water:

1. Start from the user's interacted items (seeds)
2. Expand to 1-hop KG neighbors ("ripple set")
3. Compute attention-weighted aggregation
4. Repeat for multiple hops

For user $u$ with ripple set $\mathcal{S}_u^k$ at hop $k$:

$$p_i = \text{softmax}(\mathbf{v} \cdot \mathbf{R}_i \cdot \mathbf{h}_i)$$
$$\mathbf{o}_u^k = \sum_{(h_i, r_i, t_i) \in \mathcal{S}_u^k} p_i \cdot \mathbf{t}_i$$

> **💡 Concept:** RippleNet treats each user's historical items as "seeds" and propagates preference signals through the KG. Items connected through KG relations to a user's history receive higher predicted relevance.

In [ ]:
class RippleNet(nn.Module):
    """RippleNet: Propagating User Preferences on KG (Wang et al., 2018)."""
    
    def __init__(self, n_entities, n_relations, n_items, embed_dim=64, n_hops=2):
        super().__init__()
        self.n_hops = n_hops
        self.embed_dim = embed_dim
        
        self.entity_emb = nn.Embedding(n_entities, embed_dim)
        self.relation_emb = nn.Embedding(n_relations, embed_dim * embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)
        
        # Transform layers for each hop
        self.transforms = nn.ModuleList([
            nn.Linear(embed_dim, embed_dim) for _ in range(n_hops)
        ])
        
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
    
    def forward(self, item_ids, ripple_sets):
        """
        item_ids: (batch,) candidate item IDs
        ripple_sets: list of n_hops tuples of (heads, relations, tails) for each hop
                     Each element: (batch, n_neighbors) tensor
        """
        item_emb = self.item_emb(item_ids)  # (batch, dim)
        
        o_list = []
        for hop in range(self.n_hops):
            heads, relations, tails = ripple_sets[hop]
            
            h_emb = self.entity_emb(heads)  # (batch, n_neighbors, dim)
            r_emb = self.relation_emb(relations)  # (batch, n_neighbors, dim*dim)
            t_emb = self.entity_emb(tails)  # (batch, n_neighbors, dim)
            
            # Reshape relation to matrix
            r_mat = r_emb.view(-1, h_emb.size(1), self.embed_dim, self.embed_dim)
            
            # Attention: item_emb @ R @ h
            # (batch, 1, dim) @ (batch, n_neighbors, dim, dim) -> (batch, n_neighbors, dim)
            Rh = torch.matmul(h_emb.unsqueeze(2), r_mat).squeeze(2)  # (batch, n_neighbors, dim)
            attn = (item_emb.unsqueeze(1) * Rh).sum(dim=-1)  # (batch, n_neighbors)
            attn = F.softmax(attn, dim=-1)  # (batch, n_neighbors)
            
            # Aggregate
            o = (attn.unsqueeze(-1) * t_emb).sum(dim=1)  # (batch, dim)
            o_list.append(o)
            
            # Update item embedding for next hop
            item_emb = self.transforms[hop](item_emb + o)
        
        # Final prediction
        scores = (item_emb * self.item_emb(item_ids)).sum(dim=-1)
        return scores

print("RippleNet model defined.")

## 5. KG-Enhanced Recommender: Combining KG and CF

A practical approach: use pre-trained KG entity embeddings as **item features** in a recommendation model.

In [ ]:
class KGEnhancedRec(nn.Module):
    """Simple KG-enhanced recommender: CF embeddings + KG entity embeddings."""
    
    def __init__(self, n_users, n_items, cf_dim=32, kg_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, cf_dim)
        self.item_emb = nn.Embedding(n_items, cf_dim)
        
        # KG entity embedding for items (will be initialized from pre-trained TransE)
        self.item_kg_emb = nn.Embedding(n_items, kg_dim)
        
        # Projection to combine CF and KG
        self.fusion = nn.Sequential(
            nn.Linear(cf_dim + kg_dim, cf_dim),
            nn.ReLU(),
            nn.Linear(cf_dim, cf_dim)
        )
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    
    def init_kg_emb(self, pretrained_entity_emb, item_ids=None):
        """Initialize item KG embeddings from pre-trained TransE/RotatE."""
        with torch.no_grad():
            if item_ids is None:
                item_ids = torch.arange(self.item_kg_emb.num_embeddings)
            self.item_kg_emb.weight.data = pretrained_entity_emb[item_ids].clone()
    
    def get_item_repr(self, item_ids):
        """Get fused item representation."""
        cf = self.item_emb(item_ids)
        kg = self.item_kg_emb(item_ids)
        fused = self.fusion(torch.cat([cf, kg], dim=-1))
        return fused
    
    def forward(self, user_ids, item_ids):
        user = self.user_emb(user_ids)
        item = self.get_item_repr(item_ids)
        return (user * item).sum(dim=-1)

# Initialize with pre-trained KG embeddings
kg_rec = KGEnhancedRec(data["n_users"], data["n_items"], cf_dim=32, kg_dim=64)
kg_rec.init_kg_emb(transe.entity_emb.weight.data, torch.arange(data["n_items"]))

test_u = torch.tensor([0, 1, 2])
test_i = torch.tensor([5, 10, 15])
scores = kg_rec(test_u, test_i)
print(f"KG-enhanced rec scores: {scores.detach().numpy()}")

In [ ]:
# Train KG-Enhanced Recommender
class CFDataset(Dataset):
    def __init__(self, interactions, n_items, user_items):
        self.interactions = interactions
        self.n_items = n_items
        self.user_items = user_items
    
    def __len__(self):
        return len(self.interactions)
    
    def __getitem__(self, idx):
        u, pos_i = self.interactions[idx]
        neg_i = random.randint(0, self.n_items - 1)
        while neg_i in self.user_items[u]:
            neg_i = random.randint(0, self.n_items - 1)
        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(pos_i, dtype=torch.long),
            torch.tensor(neg_i, dtype=torch.long),
        )

# Split data
train_interactions = data["interactions"][:4000]
test_interactions = data["interactions"][4000:]

train_user_items = defaultdict(set)
for u, i in train_interactions:
    train_user_items[u].add(i)

cf_dataset = CFDataset(train_interactions, data["n_items"], train_user_items)
cf_loader = DataLoader(cf_dataset, batch_size=256, shuffle=True)

# Train
torch.manual_seed(SEED)
kg_rec = KGEnhancedRec(data["n_users"], data["n_items"], cf_dim=32, kg_dim=64)
kg_rec.init_kg_emb(transe.entity_emb.weight.data, torch.arange(data["n_items"]))

optimizer = torch.optim.Adam(kg_rec.parameters(), lr=0.001)
losses = []

for epoch in range(15):
    kg_rec.train()
    total_loss = 0
    n_b = 0
    for user, pos_item, neg_item in cf_loader:
        pos_score = kg_rec(user, pos_item)
        neg_score = kg_rec(user, neg_item)
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_b += 1
    losses.append(total_loss / n_b)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/15 — Loss: {losses[-1]:.4f}")

In [ ]:
# Evaluate and compare with pure CF baseline
@torch.no_grad()
def evaluate_model(model, test_interactions, train_user_items, n_items, k=20, use_kg=True):
    model.eval()
    hits = 0
    total = 0
    
    test_by_user = defaultdict(list)
    for u, i in test_interactions:
        test_by_user[u].append(i)
    
    for u, test_items in test_by_user.items():
        all_items = torch.arange(n_items)
        user_tensor = torch.full((n_items,), u, dtype=torch.long)
        scores = model(user_tensor, all_items)
        
        # Mask training items
        for ti in train_user_items[u]:
            scores[ti] = -float("inf")
        
        _, topk = scores.topk(k)
        topk_set = set(topk.tolist())
        for ti in test_items:
            if ti in topk_set:
                hits += 1
            total += 1
    
    return hits / total

hr_kg = evaluate_model(kg_rec, test_interactions, train_user_items, data["n_items"])
print(f"KG-Enhanced Rec HR@20: {hr_kg:.4f}")

# Train pure CF baseline
class PureCF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    
    def forward(self, user_ids, item_ids):
        return (self.user_emb(user_ids) * self.item_emb(item_ids)).sum(dim=-1)

torch.manual_seed(SEED)
pure_cf = PureCF(data["n_users"], data["n_items"], embed_dim=32)
opt_cf = torch.optim.Adam(pure_cf.parameters(), lr=0.001)
for epoch in range(15):
    pure_cf.train()
    for user, pos_item, neg_item in cf_loader:
        pos_s = pure_cf(user, pos_item)
        neg_s = pure_cf(user, neg_item)
        loss = -torch.log(torch.sigmoid(pos_s - neg_s) + 1e-8).mean()
        opt_cf.zero_grad()
        loss.backward()
        opt_cf.step()

hr_cf = evaluate_model(pure_cf, test_interactions, train_user_items, data["n_items"])
print(f"Pure CF HR@20: {hr_cf:.4f}")
print(f"Improvement: {(hr_kg - hr_cf) / hr_cf * 100:.1f}%")

In [ ]:
# Visualization: training loss + comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(losses) + 1), losses, marker="o", color="steelblue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BPR Loss")
axes[0].set_title("KG-Enhanced Rec Training")
axes[0].grid(True, alpha=0.3)

models = ["Pure CF", "KG-Enhanced"]
hrs = [hr_cf, hr_kg]
colors = ["coral", "steelblue"]
axes[1].bar(models, hrs, color=colors, edgecolor="black")
axes[1].set_ylabel("HR@20")
axes[1].set_title("Pure CF vs KG-Enhanced")
for i, (m, h) in enumerate(zip(models, hrs)):
    axes[1].text(i, h + 0.005, f"{h:.4f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 6. KGAT: Knowledge Graph Attention Network

**KGAT** (Wang et al., KDD 2019) applies graph attention to the combined user-item-KG graph:

$$\alpha_{ij} = \frac{\exp(\text{LeakyReLU}(\mathbf{a}^\top [\mathbf{e}_i \| \mathbf{e}_r \| \mathbf{e}_j]))}{\sum_{k \in \mathcal{N}_i} \exp(\text{LeakyReLU}(\mathbf{a}^\top [\mathbf{e}_i \| \mathbf{e}_r \| \mathbf{e}_k]))}$$

> **🔑 Pro Tip:** KGAT unifies the user-item interaction graph and the knowledge graph into a single heterogeneous graph, then applies attentive propagation. This captures both collaborative signals and knowledge-based semantics.

In [ ]:
class KGAttentionLayer(nn.Module):
    """Single attention layer for KGAT."""
    
    def __init__(self, embed_dim, relation_dim):
        super().__init__()
        self.W = nn.Linear(embed_dim, embed_dim)
        self.attn = nn.Linear(embed_dim * 2 + relation_dim, 1)
    
    def forward(self, entity_embs, relation_embs, adj_list):
        """
        entity_embs: (n_entities, embed_dim)
        relation_embs: (n_relations, relation_dim)
        adj_list: list of (head, relation, tail) index tensors
        Returns: updated entity embeddings
        """
        heads, rels, tails = adj_list
        
        h_emb = entity_embs[heads]  # (n_edges, dim)
        r_emb = relation_embs[rels]  # (n_edges, rel_dim)
        t_emb = entity_embs[tails]  # (n_edges, dim)
        
        # Attention scores
        attn_input = torch.cat([h_emb, r_emb, t_emb], dim=-1)
        attn_scores = F.leaky_relu(self.attn(attn_input).squeeze(-1), negative_slope=0.2)
        
        # Aggregate: softmax-weighted sum per head entity
        n_entities = entity_embs.size(0)
        # Use scatter for aggregation
        attn_exp = torch.exp(attn_scores)
        denom = torch.zeros(n_entities, device=entity_embs.device)
        denom.scatter_add_(0, heads, attn_exp)
        attn_norm = attn_exp / (denom[heads] + 1e-8)
        
        weighted = attn_norm.unsqueeze(-1) * self.W(t_emb)
        agg = torch.zeros_like(entity_embs)
        agg.scatter_add_(0, heads.unsqueeze(-1).expand_as(weighted), weighted)
        
        return F.leaky_relu(entity_embs + agg, negative_slope=0.2)

# Quick test
kgat_layer = KGAttentionLayer(embed_dim=64, relation_dim=64)
test_entities = torch.randn(100, 64)
test_relations = torch.randn(10, 64)
test_adj = (
    torch.randint(0, 100, (200,)),  # heads
    torch.randint(0, 10, (200,)),   # relations
    torch.randint(0, 100, (200,)),  # tails
)
updated = kgat_layer(test_entities, test_relations, test_adj)
print(f"KGAT layer output: {updated.shape}")

## 7. Analyzing KG Connectivity and Cold-Start

One key benefit of KG is cold-start: items with few interactions but rich KG connections.

In [ ]:
# Analyze KG connectivity per item
item_kg_degree = defaultdict(int)
for h, r, t in data["triples"]:
    if h < data["n_items"]:
        item_kg_degree[h] += 1
    if t < data["n_items"]:
        item_kg_degree[t] += 1

# Item interaction count
item_interaction_count = defaultdict(int)
for _, i in data["interactions"]:
    item_interaction_count[i] += 1

# Scatter plot: KG degree vs interaction count
kg_degrees = [item_kg_degree.get(i, 0) for i in range(data["n_items"])]
int_counts = [item_interaction_count.get(i, 0) for i in range(data["n_items"])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(kg_degrees, int_counts, s=10, alpha=0.5, color="steelblue")
axes[0].set_xlabel("KG Degree")
axes[0].set_ylabel("Interaction Count")
axes[0].set_title("Item KG Connectivity vs Popularity")
axes[0].grid(True, alpha=0.3)

# Cold items: low interactions but high KG degree
cold_items = [i for i in range(data["n_items"]) if int_counts[i] <= 5]
cold_kg_degrees = [kg_degrees[i] for i in cold_items]

axes[1].hist(cold_kg_degrees, bins=20, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("KG Degree")
axes[1].set_ylabel("Count")
axes[1].set_title(f"KG Degree of Cold Items (<=5 interactions, n={len(cold_items)})")

plt.tight_layout()
plt.show()

print(f"Cold items (<=5 interactions): {len(cold_items)} / {data['n_items']}")
print(f"Avg KG degree of cold items: {np.mean(cold_kg_degrees):.1f}")
print(f"Avg KG degree of all items: {np.mean(kg_degrees):.1f}")

## 8. Exercises

### 🏋️ Exercise 1: Build a KG-Enhanced Recommender with Entity Embeddings

In [ ]:
# 🏋️ Exercise 1: Multi-Relational KG-Enhanced Rec
#
# Extend KGEnhancedRec to use relation-specific projections:
# For each KG neighbor of an item, project through a relation-specific matrix,
# then aggregate to form the item's KG representation.

class MultiRelKGRec(nn.Module):
    def __init__(self, n_users, n_items, n_entities, n_relations,
                 cf_dim=32, kg_dim=64):
        super().__init__()
        # TODO: Implement
        # 1. User CF embedding
        # 2. Item CF embedding
        # 3. Entity embedding (pre-trained or learnable)
        # 4. Relation-specific projection matrices
        # 5. Aggregation function for KG neighbors
        # 6. Fusion of CF + KG representations
        pass
    
    def get_item_kg_repr(self, item_id, neighbors):
        # TODO: 
        # 1. For each neighbor (entity, relation), project entity through relation matrix
        # 2. Aggregate (mean/attention) projected neighbors
        pass
    
    def forward(self, user_ids, item_ids):
        # TODO: Score user-item pairs using fused representations
        pass

### 🏋️ Exercise 2: Train RotatE and Compare with TransE

In [ ]:
# 🏋️ Exercise 2: RotatE Training + Comparison
#
# 1. Train RotatE on the same KG data
# 2. Use RotatE embeddings to initialize KGEnhancedRec
# 3. Compare HR@20 with TransE-initialized model

# TODO: Train RotatE
# TODO: Initialize KGEnhancedRec with RotatE embeddings
# TODO: Train and evaluate
# TODO: Compare results
pass

### 🏋️ Exercise 3: Cold-Start Evaluation

In [ ]:
# 🏋️ Exercise 3: Cold-Start Analysis
#
# Evaluate KG-enhanced vs pure CF specifically on cold-start items:
# 1. Split items into "warm" (>10 interactions) and "cold" (<=5 interactions)
# 2. Evaluate HR@20 separately for test interactions involving cold items
# 3. Plot the performance gap between KG-enhanced and pure CF for cold vs warm
#
# Expected: KG-enhanced should show larger improvement on cold items.

# TODO: Implement cold-start evaluation
pass

## Summary

| Method | Year | KG Usage | Key Idea |
|--------|------|----------|----------|
| TransE/RotatE | 2013/2019 | Embedding pre-training | Translation/rotation in embedding space |
| RippleNet | 2018 | Preference propagation | Ripple-like propagation from user history |
| KGAT | 2019 | Attentive aggregation | GAT on unified user-item-KG graph |
| KGIN | 2021 | Disentangled intents | User intent decomposition via KG |

**Key Takeaways:**
1. Knowledge graphs provide structured side information that complements collaborative signals
2. TransE/RotatE pre-training creates meaningful entity embeddings that transfer to recommendation
3. RippleNet and KGAT propagate user preferences through KG structure
4. KG-enhanced methods particularly shine for cold-start items
5. The quality and density of the KG significantly impacts recommendation improvement
6. Next chapter: **Social and Heterogeneous Graphs** for recommendation